In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [4]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in tqdm(range(samples_per_genre), desc=f"Processing {genre}"):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)




import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in tqdm(wav_files):
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

100%|██████████| 500/500 [00:31<00:00, 15.83it/s]

Successfully saved 500 feature files to /kaggle/working/features/train


## 1. You ran the generate_synthetic_dataset script with samples_per_genre=50. If the script executed successfully across all 10 target genres, exactly how many .wav files should now exist in your /kaggle/working/synthetic_mashups/train/ directory tree?

In [5]:
len(glob.glob('/kaggle/working/synthetic_mashups/train/**/*.wav', recursive=True))

500

## 2. Load any of your newly generated .wav files using torchaudio.load(). Given our configuration (target_sr=22050, duration=30), what is the exact tensor shape of the resulting waveform?
Provide in tuple format, example (x,y)

In [6]:
audio, _ = torchaudio.load("/kaggle/working/synthetic_mashups/train/rock/mashup_000.wav")
audio.shape

torch.Size([2, 661500])

## 3. Using the extract_and_save_features script with n_fft=2048, hop_length=512, and n_mels=128, you converted the waveforms into .pt files. 

If you load one of these pre-computed tensors using torch.load(), what is its exact dimension shape?

In [7]:
audio = torch.load("/kaggle/working/features/train/blues/mashup_000.pt")
audio.shape

torch.Size([2, 128, 1292])

Build Your Model (The CRNN)
Goal: You will build a model that understands both what sounds are playing (using a CNN) and the rhythm of how they play over time (using an RNN).

What to Code:

Create a PyTorch class called CRNN with these exact pieces:

1. The Input Your model will take in a 1-channel Mel-spectrogram (treat it like a single-color image).

2. The CNN (Sound Feature Extractor)

Write two blocks of layers to find patterns in the audio frequencies:

Block 1: Conv2D (32 filters, $3 \times 3$ kernel, padding 1) $\rightarrow$ BatchNorm2d $\rightarrow$ ReLU $\rightarrow$ MaxPool2d ($2 \times 2$ window).

Block 2: Conv2D (64 filters, $3 \times 3$ kernel, padding 1) ---> BatchNorm2d ---> ReLU ---> MaxPool2d ($2 \times 2$ window).

3. The Bridge (Reshaping)

The CNN outputs a 3D block of data (channels, frequencies, and time). To pass this into an RNN, you must reshape it. Keep the time steps as your sequence, and flatten the channels and frequencies together into one single feature list per time step.

4. The RNN (Rhythm Tracker)

Pass your reshaped sequence into a 1-layer Bidirectional LSTM. (Set hidden_size=64 and batch_first=True).

5. The Final Prediction

The LSTM outputs data for every single time step. To get one final answer for the whole 30-second song, take the maximum value across all time steps (known as Global Max Pooling). Finally, pass that summary vector through a standard Linear layer to output the 10 genre scores.

In [8]:
class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        if feature.shape[0] == 2:
            feature = feature.mean(dim=0, keepdim=True)
        feature = feature.float()
        return feature, label

In [9]:
class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # CNN Backbone
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # RNN
        self.lstm = nn.LSTM(
            input_size=2048,   # 64 channels × 32 mel bins
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Classifier
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):

        # Step 1: CNN feature extraction
        x = self.cnn(x)

        # Shape: (B, 64, 32, T)
        b, c, f, t = x.shape

        # Step 2: Bridge to RNN
        # Move time dimension forward
        x = x.permute(0, 3, 1, 2)

        # Flatten channels + frequency
        x = x.reshape(b, t, c * f)

        # Shape: (B, T, 2048)

        # Step 3: LSTM
        x, _ = self.lstm(x)

        # Shape: (B, T, 128)

        # Step 4: Global Max Pooling over time
        x, _ = torch.max(x, dim=1)

        # Shape: (B, 128)

        # Step 5: Classification
        logits = self.fc(x)

        return logits

## 5. Your model uses a Bidirectional LSTM with input_size=2048 and hidden_size=64. Write a small snippet using sum(p.numel() for p in model.lstm.parameters() if p.requires_grad) to calculate the exact number of trainable parameters in the LSTM layer alone. What is that integer value?

In [10]:
model = CRNN()

sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

1082368